### Youtube chatboat

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6JV6EQ1oU_i4zTvvwxcitoqK0pzK0b-HnEEbiyt4hZDuw"

In [ ]:
!pip install -q youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 11.3 MB/s eta 0:00:00


In [ ]:
!pip install -q -U langchain langchain-google-genai langchain-community faiss-cpu python-dotenv youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import requests
print(requests.__version__)

2.34.2


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_1997/1706542498.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### **To get youtube video_id

In [ ]:
#function to get video_id
from urllib.parse import urlparse, parse_qs

def get_video_id(url):
    parsed_url = urlparse(url)

    if parsed_url.hostname in ["www.youtube.com", "youtube.com"]:
        return parse_qs(parsed_url.query).get("v", [None])[0]

    elif parsed_url.hostname == "youtu.be":
        return parsed_url.path.lstrip("/")

    return None


url = input("Enter YouTube URL: ")

video_id = get_video_id(url)

print("Video ID:", video_id)

Enter YouTube URL: https://www.youtube.com/watch?v=q4arcCVfY_o
Video ID: q4arcCVfY_o


### step 1a-Indexing(document ingestion)

In [ ]:
#checking that video id work properly
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

print(ytt_api.list("3yeJ_A_K52c"))

For this video (3yeJ_A_K52c) transcripts are available in the following languages:

(MANUALLY CREATED)
None

(GENERATED)
 - en ("English (auto-generated)")[TRANSLATABLE]

(TRANSLATION LANGUAGES)
 - ar ("Arabic")
 - zh-Hant ("Chinese (Traditional)")
 - nl ("Dutch")
 - fr ("French")
 - de ("German")
 - hi ("Hindi")
 - id ("Indonesian")
 - it ("Italian")
 - ja ("Japanese")
 - ko ("Korean")
 - pt ("Portuguese")
 - ru ("Russian")
 - es ("Spanish")
 - th ("Thai")
 - uk ("Ukrainian")
 - vi ("Vietnamese")


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, RequestBlocked

video_id = "3yeJ_A_K52c"

try:
    # Create an instance of YouTubeTranscriptApi
    ytt_api = YouTubeTranscriptApi()

    # Use the fetch method on the instance with the specified language
    # The fetch method returns an iterable of FetchedTranscriptSnippet objects
    transcript_list = ytt_api.fetch(video_id, languages=['en'])

    # Flatten it to plain text by accessing the 'text' attribute of each object
    transcript = " ".join(item.text for item in transcript_list)

    print(transcript)

except TranscriptsDisabled:
    print("NO caption available for this video")
except RequestBlocked:
    print("RequestBlocked: YouTube is likely blocking access from this IP (common in cloud environments).")

[music] >> Hey folks, welcome to another episode of Blood, Sweat, and Tokens. I'm Sean C. Davis and we've got another little different type of episode for you today. So, I'm on the road and I've been on the road for the last couple of weeks and Taylor and I have been chatting back and forth and we we still wanted to get something out and we were thinking about, all right, how do we do that while we're kind of in different places and what we decided was that I was going to record a series of questions and things that have been on my mind and we're going to get Taylor's take on each one of those then cut it up and turn it into kind of a special edition podcast and then next episode we will be back to our typical format back in the office back into the the grind of everyday life for me. And the interesting thing for me is that I've been largely unplugged. I And I don't mean that I've been without electronic devices, but I haven't been tethered to them and I haven't really been following t

In [ ]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[music]', start=3.274, duration=2.02), FetchedTranscriptSnippet(text='>> Hey folks, welcome to another episode of', start=10.2, duration=4.84), FetchedTranscriptSnippet(text="Blood, Sweat, and Tokens. I'm Sean C.", start=13.04, duration=4.04), FetchedTranscriptSnippet(text="Davis and we've got another little", start=15.04, duration=5.2), FetchedTranscriptSnippet(text='different type of episode for you today.', start=17.08, duration=4.96), FetchedTranscriptSnippet(text="So, I'm on the road and I've been on the", start=20.24, duration=3.72), FetchedTranscriptSnippet(text='road for the last couple of weeks and', start=22.04, duration=3.56), FetchedTranscriptSnippet(text='Taylor and I have been chatting back and', start=23.96, duration=4.4), FetchedTranscriptSnippet(text='forth and we we still wanted to get', start=25.6, duration=3.88), FetchedTranscriptSnippet(text='something out and we were thinking', start=28.36, duration=4.32),

### Step 1b-Indexing(Text spliting)

In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=100)
chunks=splitter.create_documents([transcript])

In [ ]:
len(chunks)

21

In [ ]:
chunks[0]

Document(metadata={}, page_content="[music] >> Hey folks, welcome to another episode of Blood, Sweat, and Tokens. I'm Sean C. Davis and we've got another little different type of episode for you today. So, I'm on the road and I've been on the road for the last couple of weeks and Taylor and I have been chatting back and forth and we we still wanted to get something out and we were thinking about, all right, how do we do that while we're kind of in different places and what we decided was that I was going to record a series of questions and things that have been on my mind and we're going to get Taylor's take on each one of those then cut it up and turn it into kind of a special edition podcast and then next episode we will be back to our typical format back in the office back into the the grind of everyday life for me. And the interesting thing for me is that I've been largely unplugged. I And I don't mean that I've been without electronic devices, but I haven't been tethered to them a

### step 1c&1D-Indexing(embeding generation and storing in Vector store)

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store = FAISS.from_documents(chunks, embeddings)

### step 2 Retrieval

In [ ]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":2})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a7e5fe02000>, search_kwargs={'k': 2})

In [ ]:
retriever.invoke("how transfer learning use by gpt model")

[Document(id='790b285a-0c7a-4668-b3ec-85cf38614d2a', metadata={}, page_content="you should consider looking at one of the more democratized platforms like OpenRouter, for example, where you get access to not only the frontier models, but also the much lower cost open weight models that are coming out of China. And being able to interoperate between those is mission critical. So, my answer my response to that is how do you design a system? You basically assume the risk that these things will shift and within a year's time, maybe a month's time, you sometimes need to pivot and fairly quickly, uh depending upon the economic conditions surrounding that dependency. >> Okay, and the the story I wanted to share this week was what I have been using AI for because most of most of my time off has been spent with other people and so I've been, you know, doing some Googling and research here and there just as fun, but it's just kind of goofy everyday sort of research. Um but the one place where I'

### step 3 Augmentation

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)

In [ ]:
prompt=PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']  #input varible are context and question
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
#context come from retrieved docs
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"[music] >> Hey folks, welcome to another episode of Blood, Sweat, and Tokens. I'm Sean C. Davis and we've got another little different type of episode for you today. So, I'm on the road and I've been on the road for the last couple of weeks and Taylor and I have been chatting back and forth and we we still wanted to get something out and we were thinking about, all right, how do we do that while we're kind of in different places and what we decided was that I was going to record a series of questions and things that have been on my mind and we're going to get Taylor's take on each one of those then cut it up and turn it into kind of a special edition podcast and then next episode we will be back to our typical format back in the office back into the the grind of everyday life for me. And the interesting thing for me is that I've been largely unplugged. I And I don't mean that I've been without electronic devices, but I haven't been tethered to them and I haven't really been following 

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

### step 4 -Generation

In [ ]:
answer=llm.invoke(final_prompt)
print(answer.content)

I don't know. The provided transcript does not discuss nuclear fusion.


### Building chains

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
#chaecking the workinmg of parallei chain
parallel_chain.invoke('who is Demis')

{'context': '[music] >> Hey folks, welcome to another episode of Blood, Sweat, and Tokens. I\'m Sean C. Davis and we\'ve got another little different type of episode for you today. So, I\'m on the road and I\'ve been on the road for the last couple of weeks and Taylor and I have been chatting back and forth and we we still wanted to get something out and we were thinking about, all right, how do we do that while we\'re kind of in different places and what we decided was that I was going to record a series of questions and things that have been on my mind and we\'re going to get Taylor\'s take on each one of those then cut it up and turn it into kind of a special edition podcast and then next episode we will be back to our typical format back in the office back into the the grind of everyday life for me. And the interesting thing for me is that I\'ve been largely unplugged. I And I don\'t mean that I\'ve been without electronic devices, but I haven\'t been tethered to them and I haven\'

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
#we provide user_query in  final chain
main_chain.invoke('Can you summarize the video')

'This episode of Blood, Sweat, and Tokens features a special format due to host Sean C. Davis being on the road and largely unplugged from AI news and building. He recorded a series of questions for Taylor to answer, which will be cut into this special edition podcast.\n\nSean discusses a podcast and article by David Dayen on The American Prospect, which highlights potential financial problems if AI companies fail. He explains that private equity firms, which own foundational assets like life insurance, have been providing private credit to AI startups. If these startups fail, it could jeopardize life insurance policies and premiums, leading to a massive economic impact on a state-by-state basis, similar to the "too big to fail" banks in 2008. Sean emphasizes the need to be honest about these potentially "terrifying and dismal consequences" if the AI bubble bursts.'

### Building a GUI with Streamlit

To create a user interface for our chatbot, we'll use Streamlit. Streamlit allows you to build interactive web applications purely in Python. First, let's install it.

In [ ]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 87.9 MB/s eta 0:00:00


Now, let's create a simple Streamlit application. This app will take a question from the user, pass it to your `main_chain`, and display the answer. To run this Streamlit app in Colab, you'll need to use `!streamlit run` and then click the external URL that Streamlit provides.

In [ ]:
import streamlit as st
import os

# Your existing main_chain setup (assuming it's already defined and accessible)
# If main_chain is not accessible, you might need to re-import or redefine its components here.

st.set_page_config(page_title="YouTube Chatbot")
st.title("YouTube Transcript Chatbot")

# Input field for the question
user_question = st.text_input("Ask a question about the video transcript:")

if user_question:
    with st.spinner("Getting an answer..."):
        try:
            # Invoke the main_chain with the user's question
            response = main_chain.invoke(user_question)
            st.write("**Answer:**")
            st.info(response)
        except Exception as e:
            st.error(f"An error occurred: {e}")
            st.warning("Please ensure all previous cells are run and `main_chain` is correctly defined.")

# Instructions to run the app in Colab
st.markdown("""
To run this Streamlit app, execute this cell and then click on the external URL provided by Streamlit (usually starts with `https://<random_hash>.streamlit.app`).
""")


ModuleNotFoundError: No module named 'streamlit'

In [ ]:
# This command will run the Streamlit application.
# After executing, look for a public URL in the output, usually starting with `https://<random_hash>.streamlit.app`.
# !streamlit run /content/app.py
# Note: You'll need to save the above Streamlit code to a file named 'app.py' or similar first.
# For simplicity in Colab, you can directly run the Python code that defines the Streamlit app.
# However, Streamlit typically expects a Python file.
# Let's write the Streamlit code to a temporary file and then run it.

streamlit_code = """
import streamlit as st
import os

# Ensure your `main_chain` is globally accessible or re-import/re-define necessary components
# For this example, we assume `main_chain` is defined in the Colab global scope
# and that this script will be run in the same environment.

# You might need to manually ensure `main_chain` is available or pass it.
# For a true standalone script, you'd re-initialize all LangChain components.

# Assuming main_chain and its dependencies are loaded in the Colab session
# and can be accessed by the script when executed by `streamlit run`.

st.set_page_config(page_title="YouTube Chatbot")
st.title("YouTube Transcript Chatbot")

user_question = st.text_input("Ask a question about the video transcript:")

if user_question:
    with st.spinner("Getting an answer..."):
        try:
            # Accessing main_chain from the Colab environment
            # This might require some adjustments if `main_chain` is not directly in the global scope
            # when streamlit runs the file. A more robust solution might pass it or re-initialize.
            # For a quick demo in Colab, this often works if the notebook was run sequentially.
            response = st.session_state.main_chain.invoke(user_question) # Assuming main_chain is stored in session_state
            st.write("**Answer:**")
            st.info(response)
        except Exception as e:
            st.error(f"An error occurred: {e}")
            st.warning("Please ensure all previous cells are run and `main_chain` is correctly defined and accessible.")
"""

# To make `main_chain` accessible to the streamlit script, we'll store it in st.session_state
# This requires a small modification to the notebook cell containing `main_chain` definition
# and the Streamlit app itself.

# For now, let's create a temporary file for the streamlit app
with open("app.py", "w") as f:
    f.write(streamlit_code)

# Now, run the streamlit app. You will see a public URL in the output.
# Copy and paste this URL into your browser to interact with the GUI.
!streamlit run app.py


### Combined Streamlit Chatbot GUI

This cell will perform all the steps necessary to set up and run your YouTube chatbot with a Streamlit graphical user interface. It will:

1.  Install all required Python packages.
2.  Define the core logic for extracting YouTube transcripts and setting up the LangChain components (`main_chain`).
3.  Create a Streamlit application (`app.py`) that takes a YouTube URL, processes its transcript, and allows you to ask questions about the video.
4.  Run the Streamlit application, providing a public URL for interaction.

**Instructions:**
*   Run the cell below.
*   Once the cell finishes execution, a public URL (usually starting with `https://<random_hash>.streamlit.app`) will appear in the output.
*   Click on this URL to open your chatbot's GUI in a new browser tab.
*   In the Streamlit app, enter a YouTube video URL and then ask your questions.